In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from app.core.parser import *
from app.core.loader import *
from app.reporting.history_report import *
from app.models.features import *
from app.models.train import *

import pandas as pd
import sqlite3
pd.set_option('future.no_silent_downcasting', True)


/var/folders/4x/k1x7fbgd5bl08h5y4f2kbghw0000gn/T/ipykernel_60982/2987928993.py:11: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


In [5]:
from app.reporting.history_report import *
from app.models.create_dataset import build_training_dataset

# Training process
remove_nans = True
conn = create_connection()

dataset = build_training_dataset(conn, augment=True)

conn.close() 


# create features in memory (TODO: save to DB- this has CAVEATs as features are a function of available info)


<class 'pandas.DataFrame'>
RangeIndex: 739481 entries, 0 to 739480
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   part       739481 non-null  str  
 1   orderidx   739481 non-null  int64
 2   predidx    739481 non-null  int64
 3   predqty    739481 non-null  int64
 4   pcr        739481 non-null  int64
 5   augmented  739481 non-null  int64
dtypes: int64(5), str(1)
memory usage: 39.6 MB
None


In [6]:
from app.models.features import *
from app.models.create_dataset import *

In [8]:
conn = create_connection()

df = build_training_dataset(
    conn,
)
conn.close()

bundle = model_train(dataset, 
                     holdout_weeks=2,
                     plant='arlington')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
nickname = f"'arlington'_{ts}"

saved_path = save_model(bundle, nickname)

In [23]:
bundle

{'plant': 'arlington',
 'train_span': (orderidx                  105288.000000
  predidx                   105261.000000
  predqty                        0.000000
  pcr                            0.000000
  augmented                      0.000000
  lookahead_wk                  -1.000000
  predqty_lag_qty                0.000000
  lag_ratio                      0.000000
  predqty_mean_qty               0.000000
  predqty_std_qty                0.000000
  predqty_cv                     0.000000
  predqqty_max_qty               0.000000
  orderidx_part_count          175.000000
  predqty_1wkdiff_qty       -10026.000000
  n_prior_preds                  0.000000
  first_predqty                  0.000000
  predqty_total_drift        -8400.000000
  first_to_current_ratio         0.000000
  predqty_vs_mean_ratio          0.000000
  predqty_vs_max_ratio           0.000000
  orderidx_woy                   0.000000
  orderidx_quarter               0.000000
  predidx_woy                    0.0000

In [20]:

with create_connection() as conn:
    pred_dataset = build_prediction_dataset(conn, predidxs=[105367, 105366])


In [ ]:
predict_from_bundle()

# Add the following input parameters to the @app/ui/templates/predict.html  

# Add the parameter "Prediction week(s)"; Weeks can be multiple iso weeks with the default being the current week, this requests the model to pass all future predictions from predidx included. Add an int parameter "Extend predictions (wks)" which creates dummy predictions for weeks that don't yet have a waterfall predqty; ie, if the most forward-looking prediction is for week 30, extend predictions by 2 would create estimates for week 31 and 32 as zero, allowing the model to attempt to give a "better" estimate. Save the naive predictions (predqty). Add a "save to xlsx" checkbox with toggable naming function as in @app/ui/templates/reports.html  


,part,orderidx,predidx,predqty,pcr,lookahead_wk,predqty_lag_qty,lag_ratio,predqty_mean_qty,predqty_std_qty,...,pn0,pn1,pn2,pn3,pn4,pn5,pn6,pn7,M,amount
0,24002555,105365,105366,13,1.0,-1.0,13.0,0.928571,9.382682,12.581229,...,2,4,0,0,2,5,5,5,0,127.35000
1,24002555,105367,105366,13,1.0,1.0,13.0,0.928571,9.392758,12.565095,...,2,4,0,0,2,5,5,5,0,127.35000
2,24003702,105365,105366,7,0.0,-1.0,7.0,0.875000,7.530612,3.406895,...,2,4,0,0,3,7,0,2,0,219.08000
3,24003704,105366,105366,13,0.0,0.0,13.0,0.928571,9.537313,5.293856,...,2,4,0,0,3,7,0,4,0,127.35000
4,24003704,105367,105366,13,0.0,1.0,13.0,0.928571,9.562963,5.282479,...,2,4,0,0,3,7,0,4,0,127.35000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21624,M0760544-103H,105382,105367,60,569.0,15.0,72.0,1.180328,347.562998,500.126365,...,0,7,6,0,5,4,1,0,1,4.53127
21625,M0760544-103H,105383,105367,48,569.0,16.0,48.0,0.979592,347.403826,500.041087,...,0,7,6,0,5,4,1,0,1,4.53127
21626,M0760544-103H,105384,105367,84,569.0,17.0,84.0,0.988235,347.263941,499.945072,...,0,7,6,0,5,4,1,0,1,4.53127
21627,M0760544-103H,105385,105367,36,569.0,18.0,48.0,1.297297,347.098726,499.863744,...,0,7,6,0,5,4,1,0,1,4.53127


In [30]:
predictions = predict_from_bundle(bundle, dataset)[['part', oi, pi, pq, 'est_orderqty']]
predictions

,part,orderidx,predidx,predqty,est_orderqty
0,84266353,105288,105261,0,14.0
1,84266353,105288,105261,0,14.0
2,84272708,105288,105261,0,215.0
3,84272708,105288,105261,0,215.0
4,84272716,105288,105261,0,215.0
...,...,...,...,...,...
679365,84828865,105368,105368,2,42.0
679366,85585617,105368,105368,1,83.0
679367,85585661,105368,105368,3,17.0
679368,M0694835-103H,105368,105368,192,194.0


In [ ]:
df = build_training_dataset(con)
from app.models.train import model_train
bundle_hgb = model_train(df, holdout_weeks=4, config={'obj': 'log', 'model_type': 'hgb', 'loss': 'absolute_error', 'l2': 1.0})
bundle_rf  = model_train(df, holdout_weeks=4, config={'obj': 'log', 'model_type': 'rf'})
bundle_nn  = model_train(df, holdout_weeks=4, config={'obj': 'log', 'model_type': 'nn'})

import pandas as pd
scores = pd.concat([b['scores'] for b in (bundle_hgb, bundle_rf, bundle_nn)])


KeyError: 'orderqty'

In [33]:
filt_wf  = wf[wf[pre['pi']] >= wf[pre['pi']].max()-2]
# Get the latest prediction for each date
latest_wf = filt_wf.loc[  filt_wf.groupby(['part', oi])[pi].idxmax(),
            ['part', oi, pq]]

from app.core.parser import idx_to_week
# latest_wf[oi].apply(idx_to_week)

latest_wf[['year', 'week']] = pd.DataFrame(
    latest_wf[oi].apply(idx_to_week).tolist(),
    index=latest_wf.index
)
latest_wf.drop(oi, axis=1, inplace=True)
latest_wf.pivot_table(index=['part'],
                    columns=['year', 'week'])

predqty                                                          \
year             2025                                                           
week               14    15      16     17     18     19     20     21     22   
part                                                                            
24002555         13.0   NaN    13.0    NaN    NaN    NaN    NaN    NaN    NaN   
24002621          NaN   NaN     9.0    NaN    NaN    NaN    NaN    NaN    NaN   
24003702          7.0   NaN     NaN    NaN    NaN    NaN    NaN    NaN    NaN   
24003704          NaN  13.0    13.0    NaN    NaN    NaN    NaN    NaN    NaN   
24004574          NaN   NaN     NaN    NaN    NaN    NaN    NaN    NaN    NaN   
...               ...   ...     ...    ...    ...    ...    ...    ...    ...   
87831501          NaN   NaN     4.0    NaN    NaN    NaN    NaN    NaN    4.0   
87841756          NaN   1.0     NaN    NaN    NaN    NaN    NaN    NaN    NaN   
87865960          NaN   NaN   272.0  304.0  352.0  192.0  384.0  336.0  336.0   
M0694835-103H     NaN  60.0  1344.0  192.0  192.0  192.0  192.0  180.0  216.0   
M0760544-103H     NaN  60.0   264.0  120.0   60.0   12.0   48.0   48.0   48.0   

                      ...                                                   \
year                  ...                                                    
week              23  ...     30     31     32     33     34     35     36   
part                  ...                                                    
24002555         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
24002621         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
24003702         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
24003704         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
24004574         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
...              ...  ...    ...    ...    ...    ...    ...    ...    ...   
87831501         NaN  ...    4.0    NaN    NaN    4.0    NaN    NaN    NaN   
87841756         NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
87865960       176.0  ...  304.0  336.0  304.0  304.0  288.0  304.0  256.0   
M0694835-103H  168.0  ...   84.0   84.0   96.0  132.0  132.0  120.0   24.0   
M0760544-103H   72.0  ...   48.0   60.0   48.0   72.0   72.0   84.0   36.0   

                                    
year                                
week              37     38     39  
part                                
24002555         NaN    NaN    NaN  
24002621         NaN    NaN    NaN  
24003702         NaN    NaN    NaN  
24003704         NaN    NaN    NaN  
24004574         NaN    NaN    NaN  
...              ...    ...    ...  
87831501         NaN    4.0    NaN  
87841756         NaN    NaN    NaN  
87865960       256.0  240.0  272.0  
M0694835-103H    NaN    NaN    NaN  
M0760544-103H    NaN    NaN    NaN  

[847 rows x 26 columns]